# Modul 05 · nb06 — Conversational RAG

## Masalah: RAG satu-shot tidak mengingat percakapan

Pada notebook sebelumnya (nb01–05) kita membangun pipeline RAG yang menjawab satu pertanyaan mandiri.  
Masalah muncul saat pengguna mengajukan **pertanyaan lanjutan** dalam percakapan:

| Giliran | Pertanyaan pengguna | Masalah |
|---------|---------------------|---------|
| 1 | "Berapa hari cuti tahunan?" | Aman — mandiri |
| 2 | "Kalau cuti sakit?" | **Elipsis** — "kalau" merujuk ke konteks giliran 1 |
| 3 | "Apakah perlu surat dokter?" | **Kata ganti implisit** — "itu" / "-nya" mengacu cuti sakit |
| 4 | "Bagaimana cara mengajukannya?" | **Elipsis lagi** — "-nya" mengacu pengajuan cuti |

Jika kita langsung meng-embed "Kalau cuti sakit?", hasilnya **tidak nyambung ke dokumen manapun** — embedding terlalu pendek dan ambigu tanpa konteks.  
Retrieval akan gagal atau mengembalikan dokumen yang salah.

## Solusi: memori percakapan + history-aware rewriting

1. **Memori percakapan** — simpan riwayat giliran (user ↔ asisten) agar tersedia saat menjawab pertanyaan berikutnya.
2. **History-aware rewriting** — sebelum retrieve, tulis ulang pertanyaan lanjutan menjadi pertanyaan **mandiri** (standalone) yang bisa di-embed dengan benar.

```
User (lanjutan)
      ↓
  rewrite_followup(history, follow_up)   ← menggunakan riwayat
      ↓
  pertanyaan mandiri
      ↓
  embed → retrieve → rerank
      ↓
  generate (Qwen, grounded)
      ↓
  simpan ke ConversationalMemoryManager
```

> Notebook ini dirancang untuk Google Colab dengan **GPU T4** (Qwen2.5-3B 4-bit + reranker).

In [ ]:
# Install dependencies
!pip install -q "transformers<5" "sentence-transformers>=3.0" faiss-cpu accelerate bitsandbytes

import os, sys

# Clone the course repo if running on Colab without local files
REPO = "/content/navasena-gen-ml-course"
if not os.path.exists(REPO):
    !git clone https://github.com/chmdznr/navasena-gen-ml-course.git {REPO}

# Make rag_utils importable
sys.path.append(os.path.abspath(f"{REPO}/05_rag"))

from tools.rag_utils import ConversationalMemoryManager
print("Import ConversationalMemoryManager: OK")

## Dua jenis memori + history-aware rewriting

### 1. Window memory
Menyimpan **N giliran terakhir secara verbatim**.  
Cepat dan presisi — tidak ada informasi yang hilang dalam window.  
Kelemahannya: jika percakapan panjang, konteks bisa membengkak dan melebihi batas token model.

### 2. Summary memory
Giliran yang sudah keluar dari window di-**ringkas oleh model** menjadi satu-dua kalimat.  
Ringkasan ini disertakan di awal konteks sehingga model tetap "ingat" tema percakapan tanpa harus membaca semua giliran lama.

### 3. History-aware rewriting
Sebelum retrieve, `rewrite_followup(history_context, follow_up)` meminta model untuk **mengganti kata ganti dan elipsis** dengan entitas eksplisit dari riwayat.  
Hasilnya: pertanyaan mandiri yang embedding-nya bisa cocok dengan dokumen relevan.

### Kenapa hand-roll?
Kita **tidak** menggunakan `ConversationBufferWindowMemory` atau `ConversationSummaryMemory` dari LangChain karena keduanya sudah **deprecated** di LangChain 0.3.x.  
Dengan hand-roll, kodenya **transparan, stabil, dan mudah di-debug**.  
Di akhir notebook kita tunjukkan padanan library secara jujur.

In [ ]:
# ── Corpus: 12 pasal handbook karyawan (sama seperti nb05) ──────────────────
corpus = [
    "Setiap pegawai tetap memperoleh jatah dua belas hari libur berbayar dalam satu tahun, terpisah dari hari libur nasional.",
    "Pengajuan cuti tahunan dilakukan melalui portal karyawan dan wajib mendapat persetujuan atasan langsung paling lambat tiga hari sebelumnya.",
    "Cuti sakit dapat diambil hingga sepuluh hari setiap tahun dan harus disertai surat keterangan dari dokter bila lebih dari dua hari berturut-turut.",
    "Karyawan baru menjalani masa orientasi selama tiga hari kerja pertama, mencakup pengenalan budaya perusahaan dan pelatihan keselamatan.",
    "Cuti melahirkan diberikan selama tiga bulan penuh sesuai ketentuan ketenagakerjaan yang berlaku di Indonesia.",
    "Jam kerja standar adalah pukul 09.00 hingga 17.00 dari Senin sampai Jumat, dengan satu jam istirahat untuk makan siang.",
    "Karyawan dapat bekerja dari rumah maksimal dua hari dalam seminggu setelah mendapat persetujuan dari manajer tim.",
    "Gaji dibayarkan pada tanggal 25 setiap bulan melalui transfer bank ke rekening masing-masing pegawai.",
    "Perusahaan menyediakan asuransi kesehatan yang mencakup rawat inap dan rawat jalan bagi pegawai beserta satu anggota keluarga.",
    "Lembur pada hari libur dihitung dua kali lipat dari tarif normal dan harus dicatat dalam sistem absensi sebelum dikerjakan.",
    "Pelanggaran kode etik dapat berujung pada surat peringatan hingga pemutusan hubungan kerja sesuai tingkat keparahannya.",
    "Penggantian biaya perjalanan dinas diajukan paling lambat tujuh hari setelah perjalanan dengan melampirkan bukti pembayaran yang sah.",
]

# ── Embedder + FAISS index ───────────────────────────────────────────────────
import faiss, numpy as np, torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
cvecs = embedder.encode(corpus, convert_to_numpy=True).astype("float32")
index = faiss.IndexFlatL2(cvecs.shape[1]); index.add(cvecs)

# ── Cross-encoder reranker (bge-reranker-v2-m3, from nb03) ──────────────────
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

# ── Qwen2.5-3B-Instruct in 4-bit (T4-safe) ──────────────────────────────────
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
model_name = "Qwen/Qwen2.5-3B-Instruct"
tok = AutoTokenizer.from_pretrained(model_name)
gen = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb, device_map="auto")
print("Pipeline siap:", gen.device)

In [ ]:
# ── Qwen helpers ─────────────────────────────────────────────────────────────

def qwen(prompt, max_new_tokens=160):
    messages = [{"role": "system", "content": "Jawab ringkas dalam Bahasa Indonesia."},
                {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inp = tok(text, return_tensors="pt").to(gen.device)
    out = gen.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()


def summarize_turn(old_summary, turn):                  # di-inject ke ConversationalMemoryManager
    user, assistant = turn
    return qwen(f"Ringkasan saat ini: {old_summary or '(kosong)'}\n"
                f"Perbarui ringkasan singkat dengan menambahkan giliran ini:\n"
                f"User: {user}\nAsisten: {assistant}\n\nRingkasan baru (1-2 kalimat):", 80)


def rewrite_followup(history_context, follow_up):       # history-aware query rewriting
    if not history_context.strip():
        return follow_up
    return qwen(f"Riwayat percakapan:\n{history_context}\n\nPertanyaan lanjutan: {follow_up}\n\n"
                f"Tulis ulang pertanyaan lanjutan menjadi SATU pertanyaan MANDIRI yang lengkap "
                f"(ganti kata ganti seperti 'itu','-nya' dengan entitas dari riwayat). "
                f"Keluarkan HANYA pertanyaannya:", 60)


print("Helpers didefinisikan: qwen, summarize_turn, rewrite_followup")

In [ ]:
# ── Retriever dengan reranking ────────────────────────────────────────────────

def retrieve_rerank(query, k_over=12, k_top=3):
    qv = embedder.encode([query], convert_to_numpy=True).astype("float32")
    _, idx = index.search(qv, k_over); cand = idx[0].tolist()
    scores = reranker.predict([(query, corpus[c]) for c in cand]).tolist()
    order = sorted(range(len(cand)), key=lambda i: scores[i], reverse=True)[:k_top]
    return [corpus[cand[i]] for i in order]


# window=2 agar summary aktif lebih cepat dalam demo
memory = ConversationalMemoryManager(summarize_fn=summarize_turn, window=2)


def chat_turn(follow_up):
    standalone = rewrite_followup(memory.context(), follow_up)
    chunks = retrieve_rerank(standalone)
    ctx = "\n".join(f"- {c}" for c in chunks)
    answer = qwen(f"Konteks:\n{ctx}\n\nPertanyaan: {standalone}\n\nJawab HANYA berdasarkan konteks.")
    memory.add_turn(follow_up, answer)
    return standalone, answer


print("Retriever + chat_turn siap")

## Alur satu giliran percakapan

```
pertanyaan pengguna (mungkin elipsis / kata ganti)
        │
        ▼
 rewrite_followup(memory.context(), follow_up)
        │   ← menggunakan riwayat window + ringkasan
        ▼
 pertanyaan MANDIRI yang bisa di-embed
        │
        ▼
 retrieve_rerank(standalone)   ← embed → FAISS → rerank bge-reranker-v2-m3
        │
        ▼
 top-3 chunks relevan
        │
        ▼
 qwen(konteks + pertanyaan)    ← generate grounded (do_sample=False)
        │
        ▼
 jawaban
        │
        ▼
 memory.add_turn(follow_up, answer)   ← window evicts lama → summarize_turn
```

Kunci: **retrieve memakai pertanyaan mandiri**, bukan pertanyaan lanjutan mentah —  
sehingga embedding cocok dengan dokumen korpus.

## Demo percakapan — 4 giliran

Giliran 2–4 **tidak bisa di-retrieve langsung** tanpa riwayat.  
Perhatikan kolom `(rewrite -> ...)` — di situlah history-aware rewriting bekerja.

| # | Pertanyaan mentah | Jenis ambiguitas |
|---|-------------------|------------------|
| 1 | Berapa hari cuti tahunan untuk pegawai tetap? | Mandiri ✓ |
| 2 | Kalau cuti sakit? | Elipsis — "kalau" = "berapa hari" |
| 3 | Apakah perlu surat dokter? | Kata ganti implisit — merujuk cuti sakit |
| 4 | Bagaimana cara mengajukannya? | "-nya" = pengajuan cuti |

> Catatan: rewriting dilakukan oleh model Qwen2.5-3B — hasilnya biasanya benar tapi **bisa berbeda** dari contoh komentar di kode. Yang penting: pertanyaan hasil rewrite lebih SPESIFIK daripada pertanyaan mentah, sehingga retrieval berhasil menemukan dokumen yang tepat.

In [ ]:
# ── Scripted demo: 4 giliran percakapan ──────────────────────────────────────

memory.clear()

turns = [
    "Berapa hari cuti tahunan untuk pegawai tetap?",
    "Kalau cuti sakit?",                  # target rewrite (approx): "Berapa hari cuti sakit?"
    "Apakah perlu surat dokter?",         # target rewrite (approx): "Apakah cuti sakit perlu surat keterangan dokter?"
    "Bagaimana cara mengajukannya?",      # target rewrite (approx): "Bagaimana cara mengajukan cuti?"
]

for t in turns:
    standalone, answer = chat_turn(t)
    print(f"User    : {t}")
    print(f"  (rewrite -> {standalone})")
    print(f"Asisten : {answer}\n{'─'*60}")

## Analisis hasil

### Mengapa rewriting wajib?

Coba bandingkan embedding dua pertanyaan ini secara intuitif:

- **Tanpa rewrite**: `"Kalau cuti sakit?"` → embedding pendek, ambigu → retriever bisa mengembalikan dokumen tentang cuti tahunan atau jam kerja, bukan cuti sakit
- **Setelah rewrite**: `"Berapa hari cuti sakit yang diperbolehkan?"` → embedding spesifik → retriever menemukan *"Cuti sakit dapat diambil hingga sepuluh hari..."* dengan benar

### Pola elipsis yang ditangani

| Pola | Contoh | Resolusi |
|------|--------|----------|
| Elipsis predikat | "Kalau cuti sakit?" | Tambahkan predikat dari konteks ("berapa hari") |
| Kata ganti "-nya" | "mengajukannya" | Ganti dengan objek eksplisit ("mengajukan cuti") |
| Referensi implisit | "Apakah perlu...?" | Tambahkan subjek dari giliran sebelumnya |

### Itulah inti conversational RAG

Bukan sekadar "tambahkan chat history ke prompt" — tapi **tulis ulang pertanyaan** menjadi mandiri  
**sebelum** embedding/retrieval, sehingga semantic search bekerja dengan benar.

Setelah 4 giliran dengan `window=2`, giliran pertama sudah keluar dari window dan diringkas. Perhatikan field **`summary`** / `has_summary` pada output `memory.stats()` — itu bukti path *summary memory* benar-benar aktif.

In [ ]:
# ── Stats memori + export JSON ────────────────────────────────────────────────
import json

print("Memori:", memory.stats())                      # has_summary True berarti ada giliran lama yang sudah diringkas
convo = [{"user": u, "assistant": a} for u, a in memory.turns]
json.dump({"summary": memory.summary, "window_turns": convo},
          open("/content/percakapan.json", "w"), ensure_ascii=False, indent=2)
print("Percakapan diekspor ke /content/percakapan.json")
# Catatan: /content/ di Colab bersifat sementara — unduh file sebelum sesi berakhir.

> Sel ini interaktif: setelah mengetik di kotak **Anda:**, tekan **Enter**. Qwen butuh beberapa detik untuk menjawab (tak ada indikator), jadi diam sebentar itu normal. Ajukan pertanyaan tentang dokumen HR (mis. cuti, jam kerja, gaji), bukan sapaan seperti 'halo'.

In [ ]:
# Loop ngobrol interaktif. Setelah mengetik, TEKAN ENTER untuk mengirim.
# Perintah: 'quit' (berhenti), 'stats' (lihat memori), 'clear' (kosongkan memori).
print("=== Mode ngobrol ===")
print("Coba tanya tentang kebijakan HR, mis: 'Berapa hari cuti tahunan?' lalu Enter.")
print("Ketik 'quit' untuk berhenti.\n")
while True:
    msg = input("Anda: ").strip()
    if not msg:                                  # input kosong -> minta lagi
        continue
    if msg.lower() == "quit":
        print("Sampai jumpa!"); break
    if msg.lower() == "stats":
        print("  ", memory.stats()); continue
    if msg.lower() == "clear":
        memory.clear(); print("  (memori dikosongkan)"); continue
    print("  ⏳ Qwen sedang berpikir (~beberapa detik, tanpa loading bar)...")
    standalone, answer = chat_turn(msg)
    if standalone != msg:
        print(f"  (pertanyaan ditulis ulang -> {standalone})")
    print(f"  Asisten: {answer or '(jawaban kosong — coba pertanyaan tentang dokumen HR)'}\n")

## Referensi: begini cara library standar (LangChain) — tapi APInya sedang berpindah

Padanan `rewrite_followup` kita ada di LangChain (pindah ke `langchain_classic`):

```python
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "Berdasarkan riwayat, tulis ulang pertanyaan lanjutan jadi mandiri."),
    MessagesPlaceholder("chat_history"), ("human", "{input}")])
har = create_history_aware_retriever(llm, retriever, prompt)
```

Adapun **penyimpanan memori** di LangChain sedang berpindah paradigma:

- `ConversationBufferWindowMemory` / `ConversationSummaryMemory` → **deprecated** (0.3.x)
- `RunnableWithMessageHistory` → **juga deprecated** (core 1.3.3)
- Sekarang: **LangGraph** (`InMemorySaver` + `MessagesState` + `trim_messages`)

Justru karena API library berganti ~3x dalam 2 tahun, kita **hand-roll** inti yang sederhana, stabil, dan transparan — sambil tetap paham mekanismenya.

## Ringkasan & jembatan ke nb07

### Yang sudah kita bangun di nb06

| Komponen | Teknik | File |
|----------|--------|------|
| Window memory | Verbatim N giliran terakhir | `rag_utils.py` |
| Summary memory | Qwen meringkas giliran lama | `rag_utils.py` |
| History-aware rewriting | `rewrite_followup` — resolusi elipsis + kata ganti | nb06 |
| Retrieve + rerank | bi-encoder → bge-reranker-v2-m3 | nb06 (dari nb03) |
| Grounded generation | `apply_chat_template` + `do_sample=False` | nb06 |

### Jembatan ke nb07 — Capstone: Ask-My-Document

nb07 menyatukan **semua** yang telah kita pelajari:

1. **Ingest + chunk** (nb02) — upload PDF pengguna, chunk dengan strategi terbaik
2. **Retrieve + rerank** (nb03) — FAISS + bge-reranker-v2-m3
3. **Conversational** (nb06) — memori multi-turn + history-aware rewriting
4. **Jawaban bersitasi** — setiap klaim dilengkapi nomor halaman/pasal sumber

Pengguna cukup upload satu PDF → tanya apa saja → sistem mengingat konteks percakapan.